# 4.5 텍스트 표현하기
모델의 이전 출력과 현재 입력을 섞는 식으로 반복하여 소비하는 형태의 "순환 신경망" 모델은 텍스트 분류와 생성, 그리고 자동 번역 시스템에 적용한다.

"트랜스포머"로 불리는 신경망이 대표적으로 있고, 데이터로 부터 훈련을 통해 유추되는 다량의 말뭉치만 가지고 처음부터 끝까지 신경망을 훈련시킨다.

목표는 신경망이 처리할 수 있는 숫자의 텐서로 바꾸는 것이다.

이 작업의 첫 단계는 "데이터의 차원 정보를 재정의 하는 것"이다.

## 4.5.1 텍스트를 숫자로 변환하기
'신경망으로 텍스트를 다루는 직관적인 방법 두가지'
1. 문자 단위로 한 번에 하나의 문자를 처리.
2. 단어 단위로 신경망이 바라보는 세밀한 엔티티로 개별 단어를 처리.

어떤 방법이든지 텍스트 정보를 텐서로 인코딩하는 기술은 동일.

In [1]:
import numpy as np
import torch
torch.set_printoptions(edgeitems=2, threshold=50)

In [2]:
# 구텐베르크 웹사이트에서 제인 오스틴의 "오만과 편견"을 가져와보자
with open('/content/1342-0.txt', encoding='utf8') as f:
  text = f.read()

## 4.5.2 문자 원핫 인코딩
주의할 구체적인 사항:

문자를 원핫 인코딩할 때 중요한 점은 분석 대상 텍스트에 있는 문자 집합으로 한정하여 원핫 인코딩하는 것.

예제의 경우 영어 텍스트를 로딩했으므로 ASCII를 사용한 작은 크기의 인코딩을 다루기에 안전하지만,

문자를 다 소문자로 만들어 인코딩에서의 전체 문자 종류를 줄일 수 있다.(구두점,숫자,기대하지않는 문자 종류, 표기를 빼버린다)

이런 처리가 신경망 결과에 실질적인 차이를 미칠지 여부는 작업 종류에 따라 다르다.

텍스트에서 문자를 파싱하고 각각 원핫 인코딩을 제공하면 문자의 종류 수가 벡터 크기가 되어 각 문자가 이 크기의 벡터로 표현된다.

벡터 내 각 요소는 인코딩에서 문자에 대응하는 위치를 제외하고 0으로 표현.

In [3]:
# 텍스트를 행으로 나누고 몇 줄을 가져와 보자
lines = text.split('\n')
line = lines[200]
line

'“Impossible, Mr. Bennet, impossible, when I am not acquainted with him'

In [4]:
# 행 전체의 문자를 원핫 인코딩한 문자의 총 수를 담을 텐서를 만들자
letter_t = torch.zeros(len(line), 128) # ASCII 제한인 128로 하드코딩
letter_t.shape

torch.Size([70, 128])

In [5]:
# letter_t는 행마다 원핫 인코딩된 문자 하나를 담고 있다.
# 각 행이 일치하는 문자를 표현하도록 정확한 위치에 1을 기록.(값 1 인덱스는 인코딩에서 문자의 인덱스에 대응)
# enumerate: 반복 가능한 객체 카운터 추가
# lower: 소문자 변환
# strip: 앞 뒤 공백 제거
for i, letter in enumerate(line.lower().strip()):
  letter_index = ord(letter) if ord(letter) < 128 else 0
  # 방향이 있는 쌍따옴표처럼 ASCII에 유효하지 않은 문자는 여기서 버림(0으로 설정)
  letter_t[i][letter_index] = 1 # 원핫 인코딩 수행

## 4.5.3 모든 언어를 원핫 인코딩하기
단어 단위 인코딩은 같은 식이지만 '출현된 단어로 사전'을 만들어 문장에 나오는 단어 '시퀀스에 대해 한 단어를 한 행'으로 원핫 인코딩한다.

단어가 매우 많아 인코딩 벡터가 매우 길어지면 '임베딩'을 사용하여 단어 단위로 텍스트를 효율적으로 표현하는 방법이 있다.

원핫 인코딩이 어떻게 전개될까?



In [7]:
# clean_words: 텍스트 받아 소문자 바꾸고 구두점 날리기
# "impossible, Mr. Bennet"가 포함된 line을 인수로 clean_words를 호출
def clean_words(input_str):
  punctuation = '.,;:"!?""_-'
  word_list = input_str.lower().replace('\n', ' ').split()
  word_list = [word.strip(punctuation) for word in word_list]
  return word_list

words_in_line = clean_words(line)
line, words_in_line

('“Impossible, Mr. Bennet, impossible, when I am not acquainted with him',
 ['“impossible',
  'mr',
  'bennet',
  'impossible',
  'when',
  'i',
  'am',
  'not',
  'acquainted',
  'with',
  'him'])

In [8]:
# 인코딩에서 단어를 인덱스로 매핑
word_list = sorted(set(clean_words(text)))
word2index_dict = {word: i for (i, word) in enumerate(word_list)}

len(word2index_dict), word2index_dict['impossible']

(8484, 3828)

- word2index: 단어를 키로, 정수를 값으로 가지는 사전
원핫 인코딩은 단어에 대한 인덱스를 효율적으로 찾을 용도로 사용

문장 처리 단계에서는 문장을 단어로 나누고 각각 원핫 인코딩한다면, 단어 하나당 원핫 인코딩된 벡터가 모여 텐서 하나를 만듬

In [10]:
# 빈 벡터를 만들고 문장 내 각 단어의 원핫 인코딩 값을 부여.
word_t = torch.zeros(len(words_in_line), len(word2index_dict)) # 파이토치 텐서 생성
for i, word in enumerate(words_in_line): # for 루프로 리스트에 있는 각 word 반복
  word_index = word2index_dict[word]
  word_t[i][word_index] = 1
  print('{:2} {:4} {}'.format(i, word_index, word))

  print(word_t.shape)

 0 8324 “impossible
torch.Size([11, 8484])
 1 4905 mr
torch.Size([11, 8484])
 2  891 bennet
torch.Size([11, 8484])
 3 3828 impossible
torch.Size([11, 8484])
 4 8017 when
torch.Size([11, 8484])
 5 3740 i
torch.Size([11, 8484])
 6  445 am
torch.Size([11, 8484])
 7 5054 not
torch.Size([11, 8484])
 8  247 acquainted
torch.Size([11, 8484])
 9 8094 with
torch.Size([11, 8484])
10 3619 him
torch.Size([11, 8484])


이제 tensor는 사전에 들어 있는 단어 개수인 7,261의 인코딩 공간에서 길이가 11인 한 문장을 표현.

### 두 가지 인코딩 방식 비교
1. 문자 레벨 인코딩
- 대부분 언어는 단어 수보다 문자가 훨씬 적다. 문자를 표현해 사용하면 표현할 수 있는 클래스도 몇개 되지 않음.
2. 단어 레벨 인코딩
- 매우 큰 수의 클래스를 표현, 실제 애플리케이션에는 사전에 없는 단어도 다루게 된다. 단어는 개별 문자보다 더 많은 의미를 내포하므로 단어의 표현은 자체적으로 훨씬 많은 정보를 갖게 된다.

두 방식의 차이가 극명하여 보완하는 방식으로 "바이트 쌍 인코딩"은 사전에 개별 문자를 넣고 시작해서 지정된 사전 크기에 도달할 때 까지 가장 많이 발견된 쌍을 되풀이해 사전에 넣는 식으로 해결.

*SentencePiece 토큰 방식
```
?Im|pos|s|ible|?Mr|.?|?B|en|net|,|?impossible|...| -> ?acquatinted|?with|?him
```

대부분의 경우 매핑은 단어 단위로 구분, impossible이나 이름인 Bennet 같이 `대문자로 시작하는 경우` 에는 `세부 단위를 나눔`

## 4.5.4 텍스트 임베딩
### 원핫인코딩 장단점
- 장점: 텐서에서 카테고리 데이터를 표현할 때 매우 유용
- 단점: 말뭉치 단어에 해당하는 인코딩할 아이템 수를 효과적으로 제한하기 어려우면 실패

그 때문에 중복된 단어를 제거하고 철차를 대체하여 가짓수를 줄이거나 과거 시제나 미래 시제를 구분하지 않고 하나의 토큰으로 보는 등의 방법이 필요.

"인코딩 크기를 처리할 만한 규모로 줄이고 더 이상 늘어나지 않게 하는 방법은?"
- 하나만 값이 1이고, 나머지는 0인 벡터 대신 부동소수점 수를 가지는 벡터를 사용.

100개의 부동소수점을 가진 벡터라면 엄청나게 많은 수의 단어를 표현할 수 있다.

이 방법의 핵심은 개별 단어를 100차원 공간에 매핑하여 이후 학습을 가능케 한다. ( 이 방법을 "임베딩" 이라고 함 )

하지만 의미나 문맥에 기반해 단어 사이의 거리를 두도록 배치하는 개념을 포기하고, 입력 벡터의 구조로부터 다룰만한 내용이 거의 없게 된다.

이상적인 방법으로 `비슷한 맥락에서 사용한 단어들이 가까운 거리에 배치되는 임베딩`을 만들어 사용

기본 체언(명사)나 용언(형용사)을 차원의 축을 따라 매핑하여 임베딩 공간을 구축하게 된다면, 2차원 공간을 생성할 때 각 축에 매핑하여 임베딩에 배치한다.

예를 들어 단어 임베딩을 시작하면 '사과'를 "과일"과 "붉은" 이라는 단어의 사분면에 매핑한다. 과일뿐만 아니라 꽃, 동물 등 다양한 단어에 색상을 매핑할 수 있다.

이 수동적인 작업을 자동화한다면 원문 텍스트로 구성된 다량의 말뭉치를 잘 처리하면 방금 설명한 것과 유사한 임베딩을 구축할 수 있다.

차이점을 살펴보면, 임베딩 벡터에는 100에서 1000개 구성 요소가 존재하며, 축은 기반 개념에 직접 매핑되지 않는다. 그 대신 개념적으로 유사한 단어들은 임베딩 공간의 인접 영역에 매핑되며, 임베딩 공간의 축은 임의의 부동소수점 차원이 된다.

즉, `문장에서 개념상으로 가까운 단어를 예측해낼 수 있다`는 정도이다. 그래서 아까 `원핫 인코딩된 단어`를 신경망을 사용해 `임베딩을 만들 수 있다`는 작업으로 이어질 수 있다는 원리로 이어진다.

## 4.5.5 청사진으로서의 텍스트 임베딩
임베딩이 필수적일 때: 어휘 집합 내의 많은 개체가 숫자 벡터로 표현되어야 할 경우

임베딩은 모든 카테고리 데이터에 대한 원핫 인코딩의 훌륭한 대체재가 된다. 다른 한편으로, 텍스트를 다루는 문제를 풀 때 사전에 학습된 임베딩을 개선시켜 가는 것도 일반적인 방법이다.

